In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeAlgiers
from qiskit_aer import AerSimulator
import networkx as nx
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.coupling import CouplingMap
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
import numpy as np
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti, DecomposePauliZEvolution
from qiskit.transpiler import PassManager


In [2]:
edges = [(1, 0), (1, 4), (1, 5), (2, 3), (3, 4)]
coupling_map = CouplingMap(
    edges + [e[::-1] for e in edges]
)

In [3]:
basis_gates= ["rz", "rzz", "cx", "id", "x", "sx", "swap"]

In [4]:
config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = 6
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=coupling_map)

In [5]:
qc = QuantumCircuit(6)
ham = SparsePauliOp('ZZIIZZ', 0.5)
qc.append(PauliEvolutionGate(ham), [0,1,2,3,4,5])

In [8]:
extended_swap_strat = ExtendedSwapStrategy(
    coupling_map=coupling_map,
    swap_layers=()
)

In [9]:
pm = PassManager([
    FindCommutingPauliEvolutionsMulti(), 
    CommutingGateRouter(
        extended_swap_strat,
        max_layers=0,
        perform_extra_swaps=False
    ),
    DecomposePauliZEvolution(backend.coupling_map)])
tqc = pm.run(qc)
tqc.draw()

09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - Max layers needed to apply swap decompose: 0
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - Layer 0. Sub-layers: 1. Max interaction size: 4
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 0
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - []
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - Not implementing those gates
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - [<Qubit register=(6, "q"), index=0>, <Qubit register=(6, "q"), index=1>, <Qubit register=(6, "q"), index=2>, <Qubit register=(6, "q"), index=3>, <Qubit register=(6, "q"), index=4>, <Qubit register=(6, "q"), index=5>]
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - (<Qubit register=(6, "q"), index=0>, <Qubit register=(6, "q"), index=1>, <Qubit register=(6, "q"), index=4>, <Qubit register=(6, "q"), index=5>)
09:56:59 - qiskit_qaoa.utils.transpiler_passes - INFO - [0, 1, 4, 5]
09:56:59 - qis

q_0 -> 0 ──■──────────────────────■──
         ┌─┴─┐┌───┐        ┌───┐┌─┴─┐
q_1 -> 1 ┤ X ├┤ X ├─■──────┤ X ├┤ X ├
         └───┘└─┬─┘ │      └─┬─┘└───┘
q_4 -> 4 ───────■───┼────────■───────
                    │ZZ(1)           
q_5 -> 5 ───────────■────────────────

In [10]:
pm = PassManager([
    DecomposePauliZEvolution(backend.coupling_map)])
tqc = pm.run(qc)
tqc.draw()

09:57:33 - qiskit_qaoa.utils.transpiler_passes - INFO - [<Qubit register=(6, "q"), index=0>, <Qubit register=(6, "q"), index=1>, <Qubit register=(6, "q"), index=2>, <Qubit register=(6, "q"), index=3>, <Qubit register=(6, "q"), index=4>, <Qubit register=(6, "q"), index=5>]
09:57:34 - qiskit_qaoa.utils.transpiler_passes - INFO - (<Qubit register=(6, "q"), index=0>, <Qubit register=(6, "q"), index=1>, <Qubit register=(6, "q"), index=2>, <Qubit register=(6, "q"), index=3>, <Qubit register=(6, "q"), index=4>, <Qubit register=(6, "q"), index=5>)
09:57:34 - qiskit_qaoa.utils.transpiler_passes - INFO - [0, 1, 4, 5]
09:57:34 - qiskit_qaoa.utils.transpiler_passes - INFO - Qubit: 0. Neighbours: [False, True, False, False]
09:57:34 - qiskit_qaoa.utils.transpiler_passes - INFO - Qubit: 1. Neighbours: [False, True, True]
09:57:34 - qiskit_qaoa.utils.transpiler_passes - INFO - Qubit: 4. Neighbours: [True, False, False]


q_0: ──■──────────────────────■──
     ┌─┴─┐┌───┐        ┌───┐┌─┴─┐
q_1: ┤ X ├┤ X ├─■──────┤ X ├┤ X ├
     └───┘└─┬─┘ │      └─┬─┘└───┘
q_2: ───────┼───┼────────┼───────
            │   │        │       
q_3: ───────┼───┼────────┼───────
            │   │        │       
q_4: ───────■───┼────────■───────
                │ZZ(2)           
q_5: ───────────■────────────────

In [ ]:
sum([False, True, False, False])

In [ ]:
tqc = transpile(qc, backend, optimization_level=3, basis_gates=basis_gates)
tqc.save_unitary()

In [ ]:
tqc.draw(fold=-1)

In [ ]:
res = backend.run(tqc).result()


In [ ]:
op = res.data()['unitary']

In [ ]:
qc2 = QuantumCircuit(4)
qc2.cx(0, 1)
qc2.cx(2, 1)
qc2.cx(3, 1)
qc2.rz(1, 1)
qc2.cx(3, 1)
qc2.cx(2, 1)
qc2.cx(0, 1)



In [ ]:
qc2.draw()

In [ ]:
tqc2 = transpile(qc2, backend, optimization_level=2, basis_gates=basis_gates)
tqc2.save_unitary()

In [ ]:
tqc2.draw()

In [ ]:
res2 = backend.run(tqc).result()


In [ ]:
op2 = res2.data()['unitary']

In [ ]:
np.nonzero(op2-op)

In [ ]:
from qiskit.circuit.library.standard_gates.equivalence_library import _sel


In [ ]:
_sel.get_entry(PauliEvolutionGate(SparsePauliOp('ZZ', 1)))